<a href="https://colab.research.google.com/github/mounikas-tech-AI/hybrid-rag-system/blob/main/AI/Extract_Pg_num.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# Data Extractor

This project automates the process of parsing NTSB (National Transportation Safety Board) Aviation Accident Reports in PDF format. It uses Python to extract critical metadata (Accident Number, Date & Time) and locate specific sections (e.g., Pilot Information) across multiple reports, compiling the results into a structured CSV dataset.

## Features
- **Automated Dependency Setup**: Fast installation of OCR and PDF parsing libraries.
- **Metadata Extraction**: Regular-expression-driven parsing to isolate unique incident identifiers and timestamps.
- **Dynamic Section Detection**: Scans document contents to locate which pages contain detailed personnel records.
- **Batch Processing**: Scans an entire directory of PDFs and exports a consolidated `.csv` index.

---

## 1. Setup & Dependencies
Install the required Python packages for text processing and layout-aware PDF analysis.
```

In [ ]:
pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.9/422.9 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 58.6 MB/s eta 0:00:00


```markdown
## 2. Prototype: Extracting Metadata from a Single PDF
Before processing in bulk, we verify our target patterns against a single NTSB report using `pdfplumber` to extract structured fields from the first page.
```

In [ ]:
import pdfplumber

# Path to the uploaded PDF file
pdf_path = "/content/drive/MyDrive/CP/Report_ANC22LA059_105533_2_5_2025 1_16_22 AM.pdf"

def extract_information(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        first_page = pdf.pages[0]
        text = first_page.extract_text()

        # Extracting Accident Number
        accident_number = None
        date_time = None

        for line in text.split("\n"):
            if "Accident Number:" in line:
                accident_number = line.split("Accident Number:")[1].strip()
            if "Date & Time:" in line:
                date_time = line.split("Date & Time:")[1].strip()

        return accident_number, date_time

# Extract details
accident_number, date_time = extract_information(pdf_path)

# Display results
print("Accident Number:", accident_number)
print("Date & Time:", date_time)


Accident Number: ANC22LA059
Date & Time: June 30, 2022, 18:00 Local Registration: N73AK


```markdown
## 3. Batch Extraction & Dataset Generation
This section loops through all PDFs inside our Google Drive folder, applies robust regular expressions to find metadata, dynamic page indexing to locate "Pilot Information", and outputs a clean dataset to CSV.
```

Extracting page numbers ,Accident Number,Date & Time

In [ ]:
import pdfplumber
import re
import os
import glob
import pandas as pd

# Path to the folder containing PDFs
pdf_folder = "/content/drive/MyDrive/CP"

def extract_accident_details(pdf_path):
    """Extracts Accident Number and Date & Time from the PDF with flexible regex handling."""
    accident_pattern = re.compile(r"Accident Number:\s*([A-Z0-9]+)", re.IGNORECASE)
    date_time_pattern = re.compile(r"Date\s*&\s*Time:\s*([\w\s,:]+(?:Local|UTC))", re.IGNORECASE)  # Handles 'Local' and 'UTC'

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                accident_match = accident_pattern.search(text)
                date_time_match = date_time_pattern.search(text)

                accident_number = accident_match.group(1) if accident_match else "Not Found"
                date_time = date_time_match.group(1) if date_time_match else "Not Found"

                return accident_number, date_time  # Stops after finding the first occurrence

    return "Not Found", "Not Found"

def find_pilot_information_page(pdf_path):
    """Finds the page number where 'Pilot Information' or 'Pilot Information (A1)' appears first."""
    pilot_info_pattern = re.compile(r"Pilot Information(?: \(A1\))?", re.IGNORECASE)

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text()
            if text and pilot_info_pattern.search(text):
                return page_num  # Return immediately upon finding a match

    return "Not Found"

# Get list of all PDF files in the folder
pdf_files = glob.glob(os.path.join(pdf_folder, "*.pdf"))

# Initialize a list to store extracted data
data = []

# Process each PDF in the folder
for pdf_path in pdf_files:
    pdf_name = os.path.basename(pdf_path)  # Get PDF file name
    print(f"Processing: {pdf_name}")  # Print current file being processed

    # Extract accident details
    accident_number, date_time = extract_accident_details(pdf_path)

    # Find page number containing "Pilot Information" or "Pilot Information (A1)"
    pilot_info_page = find_pilot_information_page(pdf_path)

    # Append data to list
    data.append([pdf_name, accident_number, date_time, pilot_info_page])

# Create a DataFrame
df = pd.DataFrame(data, columns=["PDF Name", "Accident Number", "Date & Time", "Pilot Information Page"])

# Display the DataFrame
print(df)

# save to a CSV file
df.to_csv("/content/drive/MyDrive/CP/accident_reports.csv", index=False)


Processing: Report_CEN23FA248_192414_2_3_2025 7_57_57 AM.pdf
Processing: Report_CEN23LA238_192359_2_3_2025 7_58_20 AM.pdf
Processing: Report_CHI01FA329_53564_2_5_2025 1_09_16 AM.pdf
Processing: Report_CEN23LA286_192566_2_3_2025 7_46_13 AM.pdf
Processing: Report_CEN23LA072_106512_2_3_2025 7_44_37 AM.pdf
Processing: Report_ERA24LA203_194192_2_5_2025 1_18_44 AM.pdf
Processing: Report_ANC22LA059_105533_2_5_2025 1_16_22 AM.pdf
Processing: Report_CEN23LA211_192277_2_3_2025 7_58_28 AM.pdf
Processing: Report_ERA22LA379_105776_2_5_2025 1_18_07 AM.pdf
Processing: Report_CEN23LA014_106135_2_5_2025 1_16_56 AM.pdf
Processing: Report_ANC23LA032_139291_2_3_2025 7_42_32 AM.pdf
Processing: Report_CEN22FA298_105410_2_3_2025 7_50_00 AM.pdf
Processing: Report_CEN23LA251_192418_2_3_2025 7_58_12 AM.pdf
Processing: Report_CEN24LA237_194531_2_5_2025 1_17_38 AM.pdf
                                            PDF Name Accident Number  \
0   Report_CEN23FA248_192414_2_3_2025 7_57_57 AM.pdf      CEN23FA248   
1  